In [ ]:
import pandas as pd

df = pd.read_csv("/workspace/nova/data.csv")

df["timeslot_datetime_from"] = pd.to_datetime(
    df["timeslot_datetime_from"]
)
df["hour"] = df["timeslot_datetime_from"].dt.hour
df["weekday"] = df["timeslot_datetime_from"].dt.weekday
df["month"] = df["timeslot_datetime_from"].dt.month

import numpy as np

df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

#list all columns in the dataframe
print(df.columns)

In [ ]:
#visualize first 5 samples, make sure to show all columns
pd.options.display.max_columns = None
df.head(n=5)

In [ ]:
#for every column, print statistics about it, including number of unique values, number of missing values, and data type, if the column is numeric, also print mean, median, and standard deviation 
for column in df.columns:
    print(f"Column: {column}")
    print(f"Data type: {df[column].dtype}")
    print(f"Number of unique values: {df[column].nunique()}")
    print(f"Number of missing values: {df[column].isnull().sum()}")
    if df[column].dtype in ["int64", "float64"]:
        print(f"Mean: {df[column].mean()}")
        print(f"Median: {df[column].median()}")
        print(f"Standard deviation: {df[column].std()}")
        print(f"Minimum: {df[column].min()}")
        print(f"Maximum: {df[column].max()}")
    print("\n")

In [ ]:
# for column timeslot_datetime_from show first and last date
print(f"First date: {df['timeslot_datetime_from'].min()}")
print(f"Last date: {df['timeslot_datetime_from'].max()}")
#print average number of samples per day, month, and year
print(f"Average number of samples per day: {df.groupby(df['timeslot_datetime_from'].dt.date).size().mean()}")
print(f"Average number of samples per month: {df.groupby(df['timeslot_datetime_from'].dt.month).size().mean()}")
print(f"Average number of samples per year: {df.groupby(df['timeslot_datetime_from'].dt.year).size().mean()}")

In [ ]:
import pandas as pd

# Ensure datetime
df["timeslot_datetime_from"] = pd.to_datetime(df["timeslot_datetime_from"])

print(f"First date: {df['timeslot_datetime_from'].min()}")
print(f"Last date: {df['timeslot_datetime_from'].max()}")

# Daily statistics
daily_counts = df.groupby(df["timeslot_datetime_from"].dt.date).size()
print(f"Average number of samples per day: {daily_counts.mean():.2f}")

# Monthly statistics (year + month is IMPORTANT!)
monthly_counts = df.groupby(
    [df["timeslot_datetime_from"].dt.year,
     df["timeslot_datetime_from"].dt.month]
).size()

print(f"Average number of samples per month: {monthly_counts.mean():.2f}")

# Yearly statistics
yearly_counts = df.groupby(
    df["timeslot_datetime_from"].dt.year
).size()

print(f"Average number of samples per year: {yearly_counts.mean():.2f}")

# Optional — show distribution summary
print("\nYear distribution:")
print(yearly_counts)

print("\nSamples per weekday:")
print(df.groupby(df["timeslot_datetime_from"].dt.weekday).size())

df.groupby(["timeslot_datetime_from"]).size().unique()
# show me all rows where timeslot_datetime_from df.groupby(["timeslot_datetime_from"]).size() is greater than 4
print(df.groupby(["timeslot_datetime_from"]).size()) 

In [ ]:
# take first 10 samples only: group samples by timeslot_datetime_from  compare values for all columns in the group
first_10_samples = df.head(10)
grouped = first_10_samples.groupby("timeslot_datetime_from")
for name, group in grouped:
    print(f"Group: {name}")
    print(group)

In [ ]:
# we suspect duplicates of full rows first print  total number of rows in the dataframe, then print number of unique rows in the dataframe, then print number of duplicate rows in the dataframe
total_rows = len(df)
unique_rows = len(df.drop_duplicates())
duplicate_rows = total_rows - unique_rows
print(f"Total number of rows: {total_rows}")
print(f"Number of unique rows: {unique_rows}")
print(f"Number of duplicate rows: {duplicate_rows}")


df.drop_duplicates().groupby(["timeslot_datetime_from"]).size().unique()
print() 

# iterate over groups of rows with the same timeslot_datetime_from and print the group if it has other than 4 rows
grouped = df.drop_duplicates().groupby("timeslot_datetime_from")
for name, group in grouped:
    if len(group) <= 1:
        print(f"Group: {name}")
        print(group)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from typing import Tuple, Dict, List


def add_time_features(df):

    ts = pd.to_datetime(df["timeslot_datetime_from"])

    df["hour"] = ts.dt.hour
    df["weekday"] = ts.dt.weekday
    df["month"] = ts.dt.month

    # Cyclic encoding
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

    df["weekday_sin"] = np.sin(2*np.pi*df["weekday"]/7)
    df["weekday_cos"] = np.cos(2*np.pi*df["weekday"]/7)

    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    # Unix time index (seconds)
    df["time_index"] = ts.astype("int64") / 1e9
    df["time_index"] = (
        df["time_index"] - df["time_index"].min()
    ) / (
        df["time_index"].max() - df["time_index"].min()
    )   
    return df

class MultiChannelDatasetBuilder:

    def __init__(
        self,
        csv_path: str,
        timestamp_col: str = "timeslot_datetime_from",
        target_col: str = "share_15_54",
        baseline_col: str = "share_15_54_3mo_mean",
        channel_col: str = "channel_id",
        id_cols: List[str] = ["main_ident", "timeslot_datetime_from", "channel_id"],
    ):
        self.csv_path = csv_path
        self.timestamp_col = timestamp_col
        self.target_col = target_col
        self.baseline_col = baseline_col
        self.channel_col = channel_col
        self.id_cols = id_cols

        self.encoder = None
        self.channels = None

    # --------------------------------------------------------
    # LOAD + CLEAN
    # --------------------------------------------------------

    def load_and_clean(self) -> pd.DataFrame:
        df = pd.read_csv(self.csv_path)

        df[self.timestamp_col] = pd.to_datetime(df[self.timestamp_col])

        # remove duplicates
        df = df.drop_duplicates(subset=self.id_cols)

        # normalize target 0–1
        df[self.target_col] = df[self.target_col] / 100.0

        return df

    # --------------------------------------------------------
    # FEATURE SPLIT
    # --------------------------------------------------------

    def _split_feature_types(self, df: pd.DataFrame):
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        text_cols = df.select_dtypes(include=["object"]).columns.tolist()

        # remove id + target + baseline from features
        remove_cols = set(self.id_cols + [self.target_col, self.baseline_col])
        numeric_cols = [c for c in numeric_cols if c not in remove_cols]
        text_cols = [c for c in text_cols if c not in remove_cols]

        return numeric_cols, text_cols

    # --------------------------------------------------------
    # ONE HOT
    # --------------------------------------------------------

    def _prepare_text_df(self, df: pd.DataFrame, text_cols: List[str]) -> pd.DataFrame:
        if not text_cols:
            return pd.DataFrame()

        df_text = df[text_cols].copy()

        for col in text_cols:
            df_text[col] = df_text[col].astype(str)
            df_text[col] = df_text[col].fillna("__missing__")

        return df_text


    def _fit_encoder(self, df: pd.DataFrame, text_cols: List[str]):
        if not text_cols:
            return

        df_text = self._prepare_text_df(df, text_cols)

        self.encoder = OneHotEncoder(
            sparse_output=False,
            handle_unknown="ignore"
        )

        self.encoder.fit(df_text)


    def _transform_text(self, df: pd.DataFrame, text_cols: List[str]):
        if not text_cols:
            return np.empty((len(df), 0))

        df_text = self._prepare_text_df(df, text_cols)

        return self.encoder.transform(df_text)

    # --------------------------------------------------------
    # BUILD MULTITASK STRUCTURE
    # --------------------------------------------------------

    def build(self) -> Dict[str, np.ndarray]:

        df = self.load_and_clean()
        df = add_time_features(df)

        numeric_cols, text_cols = self._split_feature_types(df)

        self.channels = sorted(df[self.channel_col].unique())

        self._fit_encoder(df, text_cols)

        all_samples = []

        # group by timestamp (ensures same timestamp stays together)
        grouped = df.groupby(self.timestamp_col)

        for timestamp, group in grouped:

            channel_map = {row[self.channel_col]: row for _, row in group.iterrows()}

            for channel in self.channels:

                if channel not in channel_map:
                    continue

                row = channel_map[channel]

                # context features
                context_numeric = row[numeric_cols].values.astype(float)

                context_text = self._transform_text(
                    pd.DataFrame([row[text_cols]]) if text_cols else pd.DataFrame(),
                    text_cols
                )

                context_features = np.concatenate(
                    [context_numeric.reshape(1, -1), context_text],
                    axis=1
                )

                # competition features
                competition_features = []
                competition_mask = []

                for other_channel in self.channels:
                    if other_channel == channel:
                        continue

                    if other_channel in channel_map:
                        other_row = channel_map[other_channel]
                        comp_numeric = other_row[numeric_cols].values.astype(float)
                        competition_mask.append(1.0)
                    else:
                        comp_numeric = np.zeros(len(numeric_cols))
                        competition_mask.append(0.0)

                    competition_features.append(comp_numeric)

                competition_features = np.concatenate(competition_features)
                competition_mask = np.array(competition_mask, dtype=float)

                sample = {
                    "timestamp": timestamp,
                    "channel": channel,
                    "context_features": context_features.flatten(),
                    "competition_features": competition_features,
                    "competition_mask": competition_mask,
                    "target": row[self.target_col],
                    "baseline": row[self.baseline_col],
                }

                all_samples.append(sample)

        return self._to_numpy(all_samples)

    # --------------------------------------------------------
    # CONVERT TO NUMPY
    # --------------------------------------------------------

    def _to_numpy(self, samples: List[Dict]):

        X_context = np.stack([s["context_features"] for s in samples])
        X_comp = np.stack([s["competition_features"] for s in samples])
        X_comp_mask = np.stack([s["competition_mask"] for s in samples])
        y = np.array([s["target"] for s in samples])
        baseline = np.array([s["baseline"] for s in samples])
        timestamps = np.array([s["timestamp"] for s in samples])

        return {
            "X_context": X_context,
            "X_competition": X_comp,
            "X_competition_mask": X_comp_mask,
            "y": y,
            "baseline": baseline,
            "timestamps": timestamps,
        }

    # --------------------------------------------------------
    # TIME SERIES SPLIT
    # --------------------------------------------------------

    def time_split(
        self,
        data: Dict[str, np.ndarray],
        train_ratio=0.7,
        val_ratio=0.15,
    ) -> Tuple[Dict, Dict, Dict]:

        unique_ts = np.sort(np.unique(data["timestamps"]))

        n = len(unique_ts)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        train_ts = unique_ts[:train_end]
        val_ts = unique_ts[train_end:val_end]
        test_ts = unique_ts[val_end:]

        def subset(ts_set):
            mask = np.isin(data["timestamps"], ts_set)
            return {k: v[mask] for k, v in data.items()}

        return subset(train_ts), subset(val_ts), subset(test_ts)

In [ ]:
builder = MultiChannelDatasetBuilder("/workspace/nova/data.csv")

data = builder.build()

train, val, test = builder.time_split(data)

print(train["X_context"].shape)
print(train["X_competition"].shape)
print(train["y"].shape)



In [ ]:
print(val["X_context"].shape)
print(val["X_competition"].shape)
print(val["y"].shape)

print(test["X_context"].shape)
print(test["X_competition"].shape)
print(test["y"].shape)

In [ ]:
import torch
from torch.utils.data import Dataset


class MultiChannelTorchDataset(Dataset):

    def __init__(
        self,
        data_dict: dict,
        residual: bool = False,
        device: str = "cpu",
    ):
        """
        data_dict: výstup z builder.build() nebo time_split()
        residual: pokud True → target = y - baseline
        """

        self.X_context = torch.tensor(
            data_dict["X_context"], dtype=torch.float32
        )

        self.X_competition = torch.tensor(
            data_dict["X_competition"], dtype=torch.float32
        )

        self.X_competition_mask = torch.tensor(
            data_dict["X_competition_mask"], dtype=torch.float32
        )

        self.baseline = torch.tensor(
            data_dict["baseline"], dtype=torch.float32
        ).unsqueeze(-1)

        y = torch.tensor(data_dict["y"], dtype=torch.float32).unsqueeze(-1)

        if residual:
            self.y = y - self.baseline
        else:
            self.y = y

        self.device = device
        self.to(device)

    # --------------------------------------------------

    def to(self, device):
        self.X_context = self.X_context.to(device)
        self.X_competition = self.X_competition.to(device)
        self.X_competition_mask = self.X_competition_mask.to(device)
        self.baseline = self.baseline.to(device)
        self.y = self.y.to(device)

    # --------------------------------------------------

    def __len__(self):
        return len(self.y)

    # --------------------------------------------------

    def __getitem__(self, idx):

        return {
            "context": self.X_context[idx],
            "competition": self.X_competition[idx],
            "competition_mask": self.X_competition_mask[idx],
            "baseline": self.baseline[idx],
            "target": self.y[idx],
        }

In [ ]:
from torch.utils.data import DataLoader

builder = MultiChannelDatasetBuilder("/workspace/nova/data.csv")
data = builder.build()
train, val, test = builder.time_split(data)

train_ds = MultiChannelTorchDataset(train, residual=False)
val_ds   = MultiChannelTorchDataset(val, residual=False)
test_ds  = MultiChannelTorchDataset(test, residual=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np


class MultiChannelSequenceDataset(Dataset):
    """
    Creates sliding window sequences for LSTM / Transformer.

    Input:
        base_data: dict from builder.time_split()
        sequence_length: history length
        residual: if True -> target = y - baseline
    """

    def __init__(
        self,
        base_data: dict,
        sequence_length: int = 24,
        residual: bool = False,
        device: str = "cpu",
    ):

        self.sequence_length = sequence_length
        self.device = device

        # numpy arrays
        X_context = base_data["X_context"]
        X_comp = base_data["X_competition"]
        X_mask = base_data["X_competition_mask"]
        y = base_data["y"]
        baseline = base_data["baseline"]
        timestamps = base_data["timestamps"]

                # Normalize target to 0-1 (since share is 0-100)
        y = y / 100.0
        baseline = baseline / 100.0

        # sort by time (critical!)
        order = np.argsort(timestamps)

        X_context = X_context[order]
        X_comp = X_comp[order]
        X_mask = X_mask[order]
        y = y[order]
        baseline = baseline[order]
        timestamps = timestamps[order]

        if residual:
            y = y - baseline

        self.samples = []

        n = len(y)

        for i in range(sequence_length, n):

            # skip if timestamp gap (optional safety)
            if timestamps[i] == timestamps[i - 1]:
                continue

            seq_slice = slice(i - sequence_length, i)

            sample = {
                "context": torch.tensor(
                    X_context[seq_slice], dtype=torch.float32
                ),
                "competition": torch.tensor(
                    X_comp[seq_slice], dtype=torch.float32
                ),
                "competition_mask": torch.tensor(
                    X_mask[seq_slice], dtype=torch.float32
                ),
                "baseline": torch.tensor(
                    baseline[i], dtype=torch.float32
                ).unsqueeze(0),
                "target": torch.tensor(
                    y[i], dtype=torch.float32
                ).unsqueeze(0),
            }

            self.samples.append(sample)

        # move to device
        for s in self.samples:
            for k in s:
                s[k] = s[k].to(device)

    # -----------------------------------------------------

    def __len__(self):
        return len(self.samples)

    # -----------------------------------------------------

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
from torch.utils.data import DataLoader

builder = MultiChannelDatasetBuilder("/workspace/nova/data.csv")
data = builder.build()
train, val, test = builder.time_split(data)

train_ds = MultiChannelSequenceDataset(train, sequence_length=24, residual=False)
val_ds   = MultiChannelSequenceDataset(val, sequence_length=24, residual=False)
test_ds  = MultiChannelSequenceDataset(test, sequence_length=24, residual=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
batch = next(iter(train_loader))

print(batch["context"].shape)
# (B, T, F_context)

print(batch["competition"].shape)
# (B, T, F_competition)

print(batch["competition_mask"].shape)
# (B, T, num_other_channels)

print(batch["target"])
# (B, 1)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error


class WeeklyLookupBaselineModel:

    def __init__(
        self,
        history_months: int = 3
    ):
        self.history_months = history_months
        self.lookup_table = None

    # --------------------------------------------------

    def _build_time_keys(self, df):

        ts = pd.to_datetime(df["timeslot_datetime_from"])

        df = df.copy()

        df["weekday"] = ts.dt.weekday
        df["hour"] = ts.dt.hour
        df["minute"] = ts.dt.minute

        # 30 min slot index (0-47)
        df["half_hour_slot"] = df["hour"] * 2 + (df["minute"] >= 30).astype(int)

        return df

    # --------------------------------------------------

    def fit(self, df):

        df = self._build_time_keys(df)

        df["share"] = df["share_15_54"] / 100.0

        # rolling history filtering
        ts = pd.to_datetime(df["timeslot_datetime_from"])

        max_date = ts.max()
        min_date = max_date - pd.DateOffset(months=self.history_months)

        df_hist = df[ts >= min_date]

        # Build lookup table
        self.lookup_table = (
            df_hist
            .groupby([
                "channel_id",
                "weekday",
                "half_hour_slot"
            ])["share"]
            .mean()
            .reset_index()
        )

        print(
            f"Lookup table size = {len(self.lookup_table)} entries"
        )

    # --------------------------------------------------

    def _lookup_value(self, channel, weekday, slot):

        row = self.lookup_table[
            (self.lookup_table["channel_id"] == channel) &
            (self.lookup_table["weekday"] == weekday) &
            (self.lookup_table["half_hour_slot"] == slot)
        ]

        if len(row) > 0:
            return row["share"].values[0]

        # If missing → interpolate 2 nearest slots
        return self._interpolate(channel, weekday, slot)

    # --------------------------------------------------

    def _interpolate(self, channel, weekday, slot):

        candidates = self.lookup_table[
            (self.lookup_table["channel_id"] == channel) &
            (self.lookup_table["weekday"] == weekday)
        ]

        if len(candidates) == 0:
            return self.lookup_table["share"].mean()

        # nearest 2 slots
        candidates["dist"] = np.abs(
            candidates["half_hour_slot"] - slot
        )

        nearest = candidates.nsmallest(2, "dist")

        if len(nearest) == 1:
            return nearest["share"].values[0]

        w = nearest["dist"].values
        s = nearest["share"].values

        return np.sum(s / (w + 1e-6)) / np.sum(1 / (w + 1e-6))

    # --------------------------------------------------

    def predict(self, df):

        df = self._build_time_keys(df)

        preds = []

        for _, row in df.iterrows():

            pred = self._lookup_value(
                row["channel_id"],
                row["weekday"],
                row["half_hour_slot"]
            )

            preds.append(pred)

        return np.array(preds)

    # --------------------------------------------------

    def evaluate(self, df):

        df = df.copy()

        preds = self.predict(df)

        y = df["share_15_54"].values / 100.0

        mse = mean_squared_error(y, preds)
        mae = mean_absolute_error(y, preds)

        print("MSE:", mse)
        print("MAE:", mae)

        return mse, mae

In [ ]:
model = WeeklyLookupBaselineModel(history_months=3)

model.fit(train)

model.evaluate(val)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error


class SequenceLookupBaselineModel:

    def __init__(
        self,
        history_months: int = 3
    ):
        self.history_months = history_months
        self.lookup_table = None

    # --------------------------------------------------

    def fit(self, train_loader):

        all_records = []

        for batch in train_loader:

            # We only need last timestep from sequence
            # since target corresponds to t
            batch_size = batch["target"].shape[0]

            for i in range(batch_size):

                record = {
                    "target": batch["target"][i].cpu().numpy()[0],
                    "baseline": batch["baseline"][i].cpu().numpy()[0]
                }

                # Recover time / channel if stored in dataset
                # (if you add them later)

                all_records.append(record)

        df = pd.DataFrame(all_records)

        df["target"] = df["target"] *100.0

        # Global mean baseline lookup
        self.global_mean = df["target"].mean()

        print(f"Global mean baseline = {self.global_mean:.4f}")

    # --------------------------------------------------

    def predict_batch(self, batch):

        batch_size = batch["target"].shape[0]

        return np.ones((batch_size, 1)) * self.global_mean

    # --------------------------------------------------

    def predict(self, loader):

        preds = []
        targets = []

        for batch in loader:

            p = self.predict_batch(batch)

            preds.append(p)
            targets.append(
                batch["target"].cpu().numpy() / 100.0
            )

        return np.concatenate(preds), np.concatenate(targets)

    # --------------------------------------------------

    def evaluate(self, loader):

        preds, targets = self.predict(loader)

        mse = mean_squared_error(targets, preds)
        mae = mean_absolute_error(targets, preds)

        print("MSE:", mse)
        print("MAE:", mae)

        return mse, mae

In [ ]:
model = SequenceLookupBaselineModel(history_months=3)

model.fit(train_loader)

model.evaluate(test_loader)

In [ ]:
import torch
import torch.nn as nn


class LSTMSharePredictor(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()   # since target is 0-1
        )

    # --------------------------------------------------

    def forward(self, x):

        # x = (B, T, F)

        out, (h, c) = self.lstm(x)

        # take last timestep
        last_hidden = out[:, -1, :]

        return self.head(last_hidden)

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device):

    model.train()
    total_loss = 0

    for batch in loader:

        context = batch["context"]
        comp = batch["competition"]

        # concatenate all features
        x = torch.cat([context, comp], dim=-1)

        y = batch["target"]

        pred = model(x)

        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(model, loader, device):

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for batch in loader:

            x = torch.cat(
                [batch["context"], batch["competition"]],
                dim=-1
            )

            p = model(x)

            preds.append(p.cpu())
            targets.append(batch["target"].cpu())

    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    from sklearn.metrics import mean_absolute_error

    print("MAE:", mean_absolute_error(targets, preds))

    return preds, targets

In [ ]:
def evaluate(model, loader, device):

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for batch in loader:

            x = torch.cat(
                [batch["context"], batch["competition"]],
                dim=-1
            )

            p = model(x)

            preds.append(p.cpu())
            targets.append(batch["target"].cpu())

    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    from sklearn.metrics import mean_absolute_error

    print("MAE:", mean_absolute_error(targets, preds))

    return preds, targets

In [ ]:
context_dim = train_ds[0]["context"].shape[-1]
comp_dim = train_ds[0]["competition"].shape[-1]

model = LSTMSharePredictor(
    input_dim=context_dim + comp_dim
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [ ]:
import torch
from torch.utils.data import DataLoader


def train_model(
    model,
    train_ds,
    val_ds,
    device="cpu",
    batch_size=64,
    lr=1e-3,
    epochs=1
):

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False
    )

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()

    # --------------------------------------------------
    # Training loop
    # --------------------------------------------------

    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for batch in train_loader:

            # concatenate features
            x = torch.cat(
                [batch["context"], batch["competition"]],
                dim=-1
            ).to(device)

            y = batch["target"].to(device)

            pred = model(x)

            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # --------------------------------------------------
        # Validation
        # --------------------------------------------------

        model.eval()
        val_loss = 0

        with torch.no_grad():

            for batch in val_loader:

                x = torch.cat(
                    [batch["context"], batch["competition"]],
                    dim=-1
                ).to(device)

                y = batch["target"].to(device)

                pred = model(x)

                val_loss += loss_fn(pred, y).item()

        val_loss /= len(val_loader)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"
print("Using device:", device)

train_model(
    model,
    train_ds,
    val_ds,
    device=device,
    epochs=1
)

In [ ]:
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error


class BaseShareModel:

    def fit(self, train_loader):
        raise NotImplementedError

    def predict(self, loader):
        raise NotImplementedError

    def evaluate(self, loader):

        preds = []
        targets = []
        with torch.no_grad():
            for batch in loader:
                
                p, t = self.predict_batch(batch)

                preds.append(p)
                targets.append(t)

        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        # Remove NaN values
        mask = np.isfinite(preds) & np.isfinite(targets)

        preds = preds[mask]
        targets = targets[mask]

        from sklearn.metrics import mean_squared_error, mean_absolute_error

        mse = mean_squared_error(targets, preds)
        mae = mean_absolute_error(targets, preds)

        print("MSE:", mse)
        print("MAE:", mae)

        return mse, mae

    def to(self, device):
        # Override in torch models
        return self

In [ ]:
import numpy as np
import pandas as pd


class LookupBaselineModel(BaseShareModel):

    def __init__(self):
        self.global_mean = None

    # --------------------------------------------------

    def fit(self, train_loader):

        all_targets = []

        for batch in train_loader:
            all_targets.append(
                batch["target"].cpu().numpy()
            )

        all_targets = np.concatenate(all_targets)

        self.global_mean = np.mean(all_targets)

        print("Baseline mean:", self.global_mean)

    # --------------------------------------------------

    def predict(self, loader):

        preds = []
        targets = []

        for batch in loader:

            batch_size = batch["target"].shape[0]

            p = np.ones((batch_size, 1)) * self.global_mean

            preds.append(p)
            targets.append(
                batch["target"].cpu().numpy()
            )

        return np.concatenate(preds), np.concatenate(targets)

In [ ]:
import torch
import torch.nn as nn


class LSTMShareModel(BaseShareModel):

    def __init__(self, input_dim, hidden_dim=128):

        self.device = "cpu"

        self.model = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

        self.model.to(self.device)
        self.head.to(self.device)

        self.optimizer = torch.optim.Adam(
            list(self.model.parameters()) +
            list(self.head.parameters()),
            lr=1e-3
        )

        self.loss_fn = nn.MSELoss()

    # --------------------------------------------------

    def to(self, device):

        self.device = device

        self.model.to(device)
        self.head.to(device)

        return self

    # --------------------------------------------------

    def fit(self, train_loader):

        self.model.train()

        for batch in train_loader:
            print(batch)
            x = torch.cat(
                [batch["context"], batch["competition"]],
                dim=-1
            ).to(self.device)

            y = batch["target"].to(self.device)

            out, _ = self.model(x)
            out = self.head(out[:, -1, :])

            loss = self.loss_fn(out, y)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

    # --------------------------------------------------
    def predict_batch(self, batch):

        x = torch.cat(
            [batch["context"], batch["competition"]],
            dim=-1
        ).to(self.device)

        y = batch["target"].cpu().numpy()

        out, _ = self.model(x)
        out = self.head(out[:, -1, :])

        return out.cpu().numpy(), y
    
    def predict(self, loader):

        self.model.eval()

        preds = []
        targets = []

        with torch.no_grad():

            for batch in loader:
 
                out, y = self.predict_batch(batch)

                preds.append(out)
                targets.append(y)

        return np.concatenate(preds), np.concatenate(targets)

In [ ]:
context_dim = train_ds[0]["context"].shape[-1]
comp_dim = train_ds[0]["competition"].shape[-1]

input_dim = context_dim + comp_dim

model = LSTMShareModel(input_dim=input_dim)

model.to(device)

model.fit(train_loader)

model.evaluate(test_loader)